# Table 3 evaluation on 100 samples

This notebook recomputes **Table 3: Locality analysis of repair baselines** on **100 valid samples**.

It compares:
- `Center-fan`
- `Planar + removed-part-aware`
- `Planar + largest-hole-only`

Outputs:
- `results/csv/table3_detail_all100.csv`
- `results/csv/table3_summary_all100.csv`

Run this notebook from the `evaluation/` folder.


In [1]:

from pathlib import Path
import sys
import os
import math
import json
import traceback
import pandas as pd
import numpy as np

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("NOTEBOOK_DIR =", NOTEBOOK_DIR)
print("PROJECT_ROOT =", PROJECT_ROOT)


NOTEBOOK_DIR = D:\MyJupyter\Works\3Dsegment\evaluation
PROJECT_ROOT = D:\MyJupyter\Works\3Dsegment


In [2]:

# Try a few common dataset locations. Edit manually if needed.
CANDIDATE_DATASET_ROOTS = [
    PROJECT_ROOT / "dataset",
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "datasets",
    PROJECT_ROOT / "chair_repair_dataset",
]

DATASET_ROOT = None
for p in CANDIDATE_DATASET_ROOTS:
    if p.exists():
        DATASET_ROOT = p
        break

if DATASET_ROOT is None:
    raise FileNotFoundError(
        "Could not auto-find dataset root. Please set DATASET_ROOT manually in this cell."
    )

RESULT_DIR = PROJECT_ROOT / "results" / "csv"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

DETAIL_CSV = RESULT_DIR / "table3_detail_all100.csv"
SUMMARY_CSV = RESULT_DIR / "table3_summary_all100.csv"

print("DATASET_ROOT =", DATASET_ROOT)
print("DETAIL_CSV  =", DETAIL_CSV)
print("SUMMARY_CSV =", SUMMARY_CSV)


DATASET_ROOT = D:\MyJupyter\Works\3Dsegment\dataset
DETAIL_CSV  = D:\MyJupyter\Works\3Dsegment\results\csv\table3_detail_all100.csv
SUMMARY_CSV = D:\MyJupyter\Works\3Dsegment\results\csv\table3_summary_all100.csv


In [3]:

from baselines.baseline_center_fan import repair_one_sample_center_fan
from baselines.baseline_planar_triangulation import repair_one_sample_planar
from baselines.baseline_planar_largest_hole import repair_one_sample_planar_largest_hole

BASELINES = {
    "Center-fan": repair_one_sample_center_fan,
    "Planar + removed-part-aware": repair_one_sample_planar,
    "Planar + largest-hole-only": repair_one_sample_planar_largest_hole,
}

print("Imported baselines:")
for k in BASELINES:
    print(" -", k)


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Imported baselines:
 - Center-fan
 - Planar + removed-part-aware
 - Planar + largest-hole-only


In [4]:

def is_valid_sample_dir(sample_dir: Path) -> bool:
    if not sample_dir.is_dir():
        return False
    required_any = [
        sample_dir / "M_gt.obj",
        sample_dir / "M_d.obj",
        sample_dir / "meta.json",
    ]
    return (sample_dir / "M_d.obj").exists() and any(p.exists() for p in required_any)

def natural_key(s: str):
    parts = []
    cur = ""
    is_digit = None
    for ch in s:
        if ch.isdigit():
            if is_digit is False:
                parts.append(cur)
                cur = ch
            else:
                cur += ch
            is_digit = True
        else:
            if is_digit is True:
                parts.append(int(cur))
                cur = ch
            else:
                cur += ch
            is_digit = False
    if cur:
        parts.append(int(cur) if is_digit else cur)
    return parts

sample_dirs = [p for p in DATASET_ROOT.iterdir() if is_valid_sample_dir(p)]
sample_dirs = sorted(sample_dirs, key=lambda p: natural_key(p.name))

print("Found valid sample dirs:", len(sample_dirs))
print("First 10 sample ids:", [p.name for p in sample_dirs[:10]])

TARGET_NUM_SAMPLES = 100
if len(sample_dirs) < TARGET_NUM_SAMPLES:
    raise ValueError(f"Only found {len(sample_dirs)} valid sample dirs, fewer than {TARGET_NUM_SAMPLES}.")

sample_dirs = sample_dirs[:TARGET_NUM_SAMPLES]
print("Using sample ids:", [p.name for p in sample_dirs[:10]], "...")
print("Total selected:", len(sample_dirs))


Found valid sample dirs: 100
First 10 sample ids: ['561', '760', '2101', '2168', '2210', '2269', '2398', '2549', '2594', '2634']
Using sample ids: ['561', '760', '2101', '2168', '2210', '2269', '2398', '2549', '2594', '2634'] ...
Total selected: 100


In [5]:

LOCALITY_KEYS = [
    "num_added_faces_inside_zone",
    "num_added_faces_outside_zone",
    "face_locality_ratio",
]

OPTIONAL_KEYS = [
    "num_added_faces",
    "nearest_loop_len_after",
    "improvement",
    "mean_added_face_quality",
    "min_added_face_quality",
]

def to_float_or_none(x):
    if x is None:
        return None
    try:
        xf = float(x)
        if math.isnan(xf) or math.isinf(xf):
            return None
        return xf
    except Exception:
        return None

def evaluate_one(sample_dir: Path, baseline_name: str, fn):
    row = {
        "sample_id": sample_dir.name,
        "baseline": baseline_name,
        "sample_dir": str(sample_dir),
        "ran_ok": False,
        "error_msg": "",
    }

    try:
        out = fn(sample_dir)

        if not isinstance(out, dict):
            row["error_msg"] = f"baseline returned non-dict: {type(out)}"
            return row

        row["ran_ok"] = True

        for k in LOCALITY_KEYS + OPTIONAL_KEYS:
            row[k] = to_float_or_none(out.get(k, None))

        if "success" in out:
            try:
                row["success"] = bool(out["success"])
            except Exception:
                row["success"] = None
        else:
            row["success"] = None

        return row

    except Exception as e:
        row["error_msg"] = f"{type(e).__name__}: {e}"
        row["traceback"] = traceback.format_exc()
        return row


In [6]:

rows = []

for i, sample_dir in enumerate(sample_dirs, start=1):
    print(f"[{i:03d}/{len(sample_dirs)}] sample={sample_dir.name}")
    for baseline_name, fn in BASELINES.items():
        row = evaluate_one(sample_dir, baseline_name, fn)
        rows.append(row)

df = pd.DataFrame(rows)
df.to_csv(DETAIL_CSV, index=False, encoding="utf-8-sig")

print()
print("Saved detail CSV to:", DETAIL_CSV)
print(df.groupby(["baseline", "ran_ok"]).size())
df.head()


[001/100] sample=561
[002/100] sample=760
[003/100] sample=2101
[004/100] sample=2168
[005/100] sample=2210
[006/100] sample=2269
[007/100] sample=2398
[008/100] sample=2549
[009/100] sample=2594
[010/100] sample=2634
[011/100] sample=2689
[012/100] sample=3022
[013/100] sample=3225
[014/100] sample=3278
[015/100] sample=35109
[016/100] sample=35185
[017/100] sample=35233
[018/100] sample=35388
[019/100] sample=35396
[020/100] sample=35580
[021/100] sample=35618
[022/100] sample=35691
[023/100] sample=35871
[024/100] sample=35916
[025/100] sample=35923
[026/100] sample=36006
[027/100] sample=36074
[028/100] sample=36081
[029/100] sample=36192
[030/100] sample=36317
[031/100] sample=36402
[032/100] sample=36460
[033/100] sample=36730
[034/100] sample=36731
[035/100] sample=36768
[036/100] sample=37085
[037/100] sample=37108
[038/100] sample=37157
[039/100] sample=37231
[040/100] sample=37431
[041/100] sample=37477
[042/100] sample=37498
[043/100] sample=37505
[044/100] sample=37524
[045

,sample_id,baseline,sample_dir,ran_ok,error_msg,num_added_faces_inside_zone,num_added_faces_outside_zone,face_locality_ratio,num_added_faces,nearest_loop_len_after,improvement,mean_added_face_quality,min_added_face_quality,success
0,561,Center-fan,D:\MyJupyter\Works\3Dsegment\dataset\561,True,,22.0,0.0,1.0,22.0,0.000000,0.344789,0.455772,0.236828,True
1,561,Planar + removed-part-aware,D:\MyJupyter\Works\3Dsegment\dataset\561,True,,20.0,0.0,1.0,20.0,0.000000,0.344789,0.507504,0.399904,True
2,561,Planar + largest-hole-only,D:\MyJupyter\Works\3Dsegment\dataset\561,True,,0.0,50.0,0.0,50.0,0.344789,1.360357,0.288187,0.049627,True
3,760,Center-fan,D:\MyJupyter\Works\3Dsegment\dataset\760,True,,0.0,94.0,0.0,94.0,0.000000,2.699355,0.100407,0.004581,True
4,760,Planar + removed-part-aware,D:\MyJupyter\Works\3Dsegment\dataset\760,True,,0.0,46.0,0.0,46.0,0.000000,2.699355,0.180143,0.027010,True


In [7]:

def mean_of(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return np.nan
    return float(s.mean())

summary_rows = []
for baseline_name, g in df.groupby("baseline", sort=False):
    g_ok = g[g["ran_ok"] == True].copy()

    summary_rows.append({
        "Method": baseline_name,
        "n_total": int(len(g)),
        "n_ran_ok": int(len(g_ok)),
        "Mean added faces inside zone ↑": mean_of(g_ok["num_added_faces_inside_zone"]),
        "Mean added faces outside zone ↓": mean_of(g_ok["num_added_faces_outside_zone"]),
        "Mean face locality ratio ↑": mean_of(g_ok["face_locality_ratio"]),
    })

summary_df = pd.DataFrame(summary_rows)

order = [
    "Center-fan",
    "Planar + removed-part-aware",
    "Planar + largest-hole-only",
]
summary_df["__order"] = summary_df["Method"].map({k: i for i, k in enumerate(order)})
summary_df = summary_df.sort_values("__order").drop(columns="__order").reset_index(drop=True)

summary_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")
summary_df


,Method,n_total,n_ran_ok,Mean added faces inside zone ↑,Mean added faces outside zone ↓,Mean face locality ratio ↑
0,Center-fan,100,100,37.056180,51.932584,0.714825
1,Planar + removed-part-aware,100,100,35.067416,42.303371,0.724519
2,Planar + largest-hole-only,100,100,3.910112,74.292135,0.081214


In [8]:

pretty_df = summary_df.copy()
for col in [
    "Mean added faces inside zone ↑",
    "Mean added faces outside zone ↓",
    "Mean face locality ratio ↑",
]:
    pretty_df[col] = pretty_df[col].map(lambda x: round(float(x), 4) if pd.notna(x) else x)

pretty_df


,Method,n_total,n_ran_ok,Mean added faces inside zone ↑,Mean added faces outside zone ↓,Mean face locality ratio ↑
0,Center-fan,100,100,37.0562,51.9326,0.7148
1,Planar + removed-part-aware,100,100,35.0674,42.3034,0.7245
2,Planar + largest-hole-only,100,100,3.9101,74.2921,0.0812


In [9]:

print("| Method | Mean added faces inside zone ↑ | Mean added faces outside zone ↓ | Mean face locality ratio ↑ |")
print("|---|---:|---:|---:|")
for _, r in pretty_df.iterrows():
    print(f"| {r['Method']} | {r['Mean added faces inside zone ↑']} | {r['Mean added faces outside zone ↓']} | {r['Mean face locality ratio ↑']} |")


| Method | Mean added faces inside zone ↑ | Mean added faces outside zone ↓ | Mean face locality ratio ↑ |
|---|---:|---:|---:|
| Center-fan | 37.0562 | 51.9326 | 0.7148 |
| Planar + removed-part-aware | 35.0674 | 42.3034 | 0.7245 |
| Planar + largest-hole-only | 3.9101 | 74.2921 | 0.0812 |


## Notes

- This notebook is focused on **Table 3 locality metrics only**.
- It uses the first 100 valid sample directories found under `DATASET_ROOT`, sorted naturally by folder name.
- If you want a different 100-sample subset, change the sample selection cell.
- If a baseline crashes on some sample, that sample still stays in the detail CSV, but only `ran_ok == True` rows contribute to the summary means.
